In [1]:
import pandas as pd
import nltk
import pickle
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
import tensorflow as tf
from tensorflow.keras.layers import Embedding,LSTM,Bidirectional,Dense,Dropout,GRU
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping

c:\Users\adnan\anaconda3\envs\tf-gpu\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
fake_df = pd.read_csv('Fake.csv')
real_df = pd.read_csv('True.csv')

In [4]:
fake_df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
real_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
fake_df['label'] = 1 #Fake news 1
real_df['label'] = 0 #Real news 0

In [7]:
fake_df.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [8]:
df = pd.concat([fake_df, real_df], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True) #Reshuffling the Dataframe for better exposure to the dataset

In [9]:
df.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0


In [10]:
df.shape

(44898, 5)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [12]:
df.isnull().sum() #if any null then remove it no imputation in text data

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [13]:
df['label'].value_counts()

label
1    23481
0    21417
Name: count, dtype: int64

In [14]:
df['content'] = df['title'] + " " + df['text']
df.drop('date', axis=1, inplace=True)
df.drop('subject', axis=1, inplace=True)

In [15]:
df.head(5)

,title,text,label,content
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",1,Ben Stein Calls Out 9th Circuit Court: Committ...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,0,Trump drops Steve Bannon from National Securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,0,Puerto Rico expects U.S. to lift Jones Act shi...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",1,OOPS: Trump Just Accidentally Confirmed He Le...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",0,Donald Trump heads for Scotland to reopen a go...


In [16]:
X = df['content']
y = df['label']

In [17]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

X_val_text, X_test_text, y_val, y_test = train_test_split(X_temp_text,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

In [18]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def data_preprocessing(text):

    text = str(text)

    text = re.sub(r'http\S+|www\S+|https\S+', "", text)
    text = re.sub(r'@\w+', "", text)
    text = re.sub(r'[^a-zA-z\s]', "", text)
    text = re.sub(r'\d+', "", text)

    text = text.lower()

    text = re.sub(r'\s+', " ", text).strip()

    tokens = word_tokenize(text)
    proc_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    proc_text = ' '. join(proc_tokens)
    return proc_text

In [19]:
train_corpus = [data_preprocessing(text)for text in X_train_text]
val_corpus = [data_preprocessing(text)for text in X_val_text]
test_corpus = [data_preprocessing(text)for text in X_test_text]

In [32]:
voc_size = 8000

Tokenization

In [33]:
tokenizer = Tokenizer(num_words=voc_size,oov_token='<OOV>')
tokenizer.fit_on_texts(train_corpus)

In [34]:
X_train_seq = tokenizer.texts_to_sequences(train_corpus)
X_val_seq = tokenizer.texts_to_sequences(val_corpus)
X_test_seq = tokenizer.texts_to_sequences(test_corpus)

In [35]:
sent_len=64
X_train=pad_sequences(X_train_seq,padding='post',maxlen=sent_len)
X_val = pad_sequences(X_val_seq,maxlen=sent_len,padding='post')

X_test = pad_sequences(X_test_seq,maxlen=sent_len,padding='post')

In [36]:
X_train[1]

array([   3, 2244,   10,  847,  366,   38,   50,    3,  651,   14,   11,
       1949,  252, 4310,  394,  246,    8,  552,    5,    3,    8,  651,
         38,   50,  455,  552,    1,   11, 1423,   92,    5,   24,  666,
        391,  150, 1108,   14,   31, 3699, 1000,   49, 3116, 1187, 2072,
          1,  313,   84,  821,  559, 2000,  455,  552,  892,   49,  654,
        114, 2339,  455,  552,  102,    6, 1022,    1, 3936])

In [37]:
embedding_vector_features=32 ##features representation
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_len))
model.add(Bidirectional(GRU(32)))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 64, 32)            256000    
                                                                 
 bidirectional_1 (Bidirectio  (None, 64)               12672     
 nal)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 268,737
Trainable params: 268,737
Non-trainable params: 0
_________________________________________________________________
None


In [38]:
early_stop = EarlyStopping(monitor = 'val_loss',patience=2,restore_best_weights=True)


In [39]:
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=20,batch_size=32,callbacks=[early_stop])

Epoch 1/20
1123/1123 [==============================] - 25s 20ms/step - loss: 0.1280 - accuracy: 0.9477 - val_loss: 0.0617 - val_accuracy: 0.9802
Epoch 2/20
1123/1123 [==============================] - 25s 22ms/step - loss: 0.0300 - accuracy: 0.9906 - val_loss: 0.0592 - val_accuracy: 0.9820
Epoch 3/20
1123/1123 [==============================] - 23s 20ms/step - loss: 0.0125 - accuracy: 0.9958 - val_loss: 0.0661 - val_accuracy: 0.9817
Epoch 4/20
1123/1123 [==============================] - 24s 21ms/step - loss: 0.0071 - accuracy: 0.9976 - val_loss: 0.0608 - val_accuracy: 0.9791


In [44]:
y_pred_prob=model.predict(X_test)
y_pred=(y_pred_prob >= 0.5).astype(int)
confusion_matrix(y_test,y_pred)

141/141 [==============================] - 1s 7ms/step


array([[2105,   37],
       [  49, 2299]], dtype=int64)

In [45]:
accuracy_score(y_test,y_pred)

0.9808463251670378

In [46]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2142
           1       0.98      0.98      0.98      2348

    accuracy                           0.98      4490
   macro avg       0.98      0.98      0.98      4490
weighted avg       0.98      0.98      0.98      4490



In [47]:
model.save('model_news_claf.h5')
with open('tokenizer.pkl', 'wb') as file:
    pickle.dump(tokenizer,file)